In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv("data/ForeignGifts_edu.csv")
fixed_col_names = ['id', 'opeid', 'institution_name', 'city', 'state', 'foreign_gift_received_date', 
                   'foreign_gift_amount', 'gift_type', 'country_of_giftor', 'giftor_name']
df.columns = fixed_col_names
df.gift_type.value_counts()

gift_type
Contract         17274
Monetary Gift    10936
Real Estate         11
Name: count, dtype: int64

### 2: Describe the differences between different classes of gift types
- Contract — foreign source gets something back (research services, tuition coverage for students, licensing rights), so the university owes performance rather than just gratitude.

- Monetary Gift — cash or cash-equivalent transferred with nothing owed in return.

- Real Estate — a transfer of land or buildings rather than money.

In [29]:
# 1 - 3 - 1
df.groupby('country_of_giftor')['foreign_gift_amount'].sum().nlargest(1)

country_of_giftor
QATAR    2706240869
Name: foreign_gift_amount, dtype: int64

In [30]:
# 1 - 3 - 2
df.groupby('country_of_giftor')['foreign_gift_amount'].count().sort_values(ascending=False).head(1)

country_of_giftor
ENGLAND    3655
Name: foreign_gift_amount, dtype: int64

In [31]:
# 1 - 3 - 3
df.groupby('country_of_giftor')['foreign_gift_amount'].mean().nlargest(1)

country_of_giftor
BERMUDA    7.688837e+06
Name: foreign_gift_amount, dtype: float64

In [ ]:
# 1 - 3 - 4
df.groupby('institution_name')['foreign_gift_amount'].sum().nlargest(1)

institution_name
Carnegie Mellon University    1477922504
Name: foreign_gift_amount, dtype: int64

In [33]:
# 1 - 3 - 5
df.groupby('institution_name')['foreign_gift_amount'].count().nlargest(1)

institution_name
University of California, Los Angeles    3916
Name: foreign_gift_amount, dtype: int64

Build a plot of the top 20 institutions by the amount or number of gifts they have received using either seaborn or plotly. 

In [ ]:
import plotly.express as px
top_20 = df.groupby('institution_name')['foreign_gift_amount'].sum().nlargest(20).reset_index()
fig = px.bar(
    top_20,
    x='foreign_gift_amount',
    y='institution_name',
    orientation='h',
    labels={'foreign_gift_amount': 'Total ($)', 'institution_name': ''},
    title='Top 20 Institutions by Foreign Gift and Contract Value'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
fig.show() 

In [20]:
# 1 - 3 - 6
print(round(df.groupby('institution_name')['foreign_gift_amount'].sum().mean(), 2))
print(df.groupby('institution_name')['foreign_gift_amount'].sum().median())

52202878.97
6486791.5


Plot a histogram of the amounts received by each university, and then label the median and the mean as vertical lines.

In [45]:
median_val = df.groupby('institution_name')['foreign_gift_amount'].sum().median()
mean_val = df.groupby('institution_name')['foreign_gift_amount'].sum().mean()
fig2 = px.histogram(df.groupby('institution_name')['foreign_gift_amount'].sum(), x = 'foreign_gift_amount',
    nbins = 50, 
    title='Distribution of Total Foreign Gifts by Institution',
    labels={'foreign_gift_amount': 'Total Gift Amount'}) 
fig2.add_vline(x = median_val, line_color = 'chartreuse', line_dash = 'dash',
               annotation_text = f'Median: {median_val}', annotation_position='top left')
fig2.add_vline(x = mean_val, line_color = 'red', line_dash = 'dash',
               annotation_text = f'Mean: {mean_val}', annotation_position='top right')
fig2.update_layout(yaxis_title = 'Number of Institutions')
fig2.show()

First, use `pd.crosstab` to look at the relationship between outcoming gifts from countries and incoming gifts to institutions.

In [47]:
# 1 - 3 - 7
pd.crosstab(df['country_of_giftor'], df['institution_name'])

institution_name,Adelphi University,Albert Einstein College of Medicine,Alfred University,American University (The),Amherst College,Arizona State University,Auburn University Montgomery,Babson College,Ball State University,Barnard College,...,Wilkes University,William Marshall Rice University,Williams College,Winthrop University,Worcester Polytechnic Institute,Wright State University,Xavier University of Louisiana,Yale University,Yeshiva University,Young Americans College of the Performing Arts (The)
country_of_giftor,,,,,,,,,,,,,,,,,,,,,
AFGHANISTAN,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AMERICAN SAMOA,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ANGOLA,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ANTIGUA,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ARGENTINA,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
VIRGIN ISLANDS (BRITISH),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,2,0,0
WALES,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
YEMEN,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# 1 - 3 - 7
df.groupby(['country_of_giftor', 'institution_name'])['foreign_gift_amount'].sum().nlargest(1)

country_of_giftor  institution_name  
QATAR              Cornell University    1018473315
Name: foreign_gift_amount, dtype: int64

In [50]:
import plotly.graph_objects as go

#giftor = 'Giftor Name'
giftor = 'country_of_giftor'
recipi = 'institution_name'
flow = 'foreign_gift_amount'
N = 25

flows = (
    df.groupby([giftor, 
                recipi])
      [flow]
      .sum()
      .nlargest(N)
      .reset_index()
)

labels = (
    flows[giftor].tolist()
    + flows[recipi].tolist()
)

labels = list(dict.fromkeys(labels))

fig = go.Figure(
    go.Sankey(
        node=dict(label=labels),
        link=dict(
            source=flows[giftor]
                        .map(labels.index),
            target=flows[recipi]
                        .map(labels.index),
            value=flows[flow]
        )
    )
)

fig.show()

In [51]:
# 1 - 3 - 8
df.groupby('country_of_giftor')['foreign_gift_amount'].sum().nlargest(10)

country_of_giftor
QATAR           2706240869
ENGLAND         1464906771
CHINA           1237952112
SAUDI ARABIA    1065205930
BERMUDA          899593972
CANADA           898160656
HONG KONG        887402529
JAPAN            655954776
SWITZERLAND      619899445
INDIA            539556490
Name: foreign_gift_amount, dtype: int64

# 1 - 3
3. Answer the following questions:
    1. Which country gives the most money in total?
        - QATAR    2706240869
    2. Which country initiates the most gifts by count?
        - ENGLAND    3655
    3. Which country gives the largest gifts on average?
        - BERMUDA    7.688837e+06
    4. Which institution receive the most money in total?
        - Carnegie Mellon University    1477922504
    5. Which institution receive the greatest number of gifts by count?
        - University of California, Los Angeles    3916
    6. What is the average amount each university receives? Median amount? If they are different why?
        - mean: 52202878.97
        - median : 6486791.5
        - A small number of institutions receive large aggregate amounts, pulling the mean up, while the median sits near the majority of institutions who receive far less.
    7. What is the largest flow of a single country to a single institution?
        - QATAR              Cornell University    1018473315
    8. Which are the top giftors to US academic institutions?
        1. Qatar
        2. England
        3. China
        4. Saudi Arabia
        5. Bermuda